# 2041. Accepted Candidates From the Interviews

## Problem Description

A hiring system stores candidate experience and scores from multiple interview rounds.

### Table Schema

**Candidates**

| Column | Type | Meaning / constraint |
| --- | --- | --- |
| candidate_id | int | unique candidate identifier (key) |
| name | varchar | candidate name |
| years_of_exp | int | years of experience |
| interview_id | int | interview identifier |

**Rounds**

| Column | Type | Meaning / constraint |
| --- | --- | --- |
| interview_id | int | interview identifier |
| round_id | int | round identifier within interview |
| score | int | round score |

### Task

The interview_id and round_id pair identifies a round record. Accept a candidate only if years_of_exp is at least 2 and the sum of scores for all rounds associated with the candidate's interview_id is strictly greater than 15. Return only candidate_id for accepted candidates. Any row order is accepted.

## Example

### Input

**candidates**

| candidate_id | name | years_of_exp | interview_id |
| --- | --- | --- | --- |
| 1 | A | 2 | 10 |
| 2 | B | 1 | 20 |
| 3 | C | 5 | 30 |
| 4 | D | 3 | 40 |

**rounds**

| interview_id | round_id | score |
| --- | --- | --- |
| 10 | 1 | 8 |
| 10 | 2 | 8 |
| 20 | 1 | 20 |
| 30 | 1 | 15 |
| 40 | 1 | 5 |

### Expected Output

| candidate_id |
| --- |
| 1 |

### Explanation

Candidate 1 has exactly two years of experience and total score 16, so qualifies. Candidate 2 has score 20 but insufficient experience. Candidate 3 has exactly 15 points and fails the strict threshold. Candidate 4 scores 5.

## Practice

The next code cell constructs exactly these example DataFrames. Try the blank practice cell before reading the solutions.

## References

The problem is restated for self-contained practice; the examples are original. Community links are for further reading, and implementations below are independently written.

[Doocs problem reference](https://github.com/doocs/leetcode/blob/main/solution/2000-2099/2041.Accepted%20Candidates%20From%20the%20Interviews/README_EN.md) · [LeetCode](https://leetcode.com/problems/accepted-candidates-from-the-interviews/)

[LeetCode discussions](https://leetcode.com/problems/accepted-candidates-from-the-interviews/solutions/)

In [ ]:
import pandas as pd
from pandas.testing import assert_frame_equal

candidates = pd.DataFrame([[1,"A",2,10],[2,"B",1,20],[3,"C",5,30],[4,"D",3,40]], columns=["candidate_id","name","years_of_exp","interview_id"])
rounds = pd.DataFrame([[10,1,8],[10,2,8],[20,1,20],[30,1,15],[40,1,5]], columns=["interview_id","round_id","score"])
expected = pd.DataFrame([[1]], columns=["candidate_id"])
display(candidates)
display(rounds)
display(expected)

In [ ]:
# Your solution here
# result = ...

## Solutions

### Solution 1 — Aggregate and merge
經驗門檻包含 2，分數嚴格大於 15。

In [ ]:
def solution_1(candidates, rounds):
    g = rounds.groupby('interview_id',as_index=False).agg(total=('score','sum'))
    m = candidates.merge(g,on='interview_id')
    return m.loc[m.years_of_exp.ge(2)&m.total.gt(15),['candidate_id']]

result_1 = solution_1(candidates.copy(), rounds.copy())
display(result_1)

### Solution 2 — Map total
每列回填 interview total，簡潔且不改變粒度。

In [ ]:
def solution_2(candidates, rounds):
    s = candidates.interview_id.map(rounds.groupby('interview_id').score.sum())
    return candidates.loc[candidates.years_of_exp.ge(2)&s.gt(15),['candidate_id']]

result_2 = solution_2(candidates.copy(), rounds.copy())
display(result_2)

### Solution 3 — Semi-join
先篩選達標 interview IDs，再過濾 candidates。

In [ ]:
def solution_3(candidates, rounds):
    g = rounds.groupby('interview_id').score.sum()
    return candidates.loc[candidates.years_of_exp.ge(2)&candidates.interview_id.isin(g[g.gt(15)].index),['candidate_id']]

result_3 = solution_3(candidates.copy(), rounds.copy())
display(result_3)

## Check all solutions
Rows are normalized only for the example comparison; preserve required ordering in the solutions themselves.

In [ ]:
def normalize(df):
    return df.sort_values(list(df.columns),na_position='last').reset_index(drop=True)
assert_frame_equal(normalize(result_1), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_2), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_3), normalize(expected), check_dtype=False)
print('All example checks passed')

## Review
Compare row grain, missing matches, duplicate keys and zero-count categories.